In [ ]:
pip install psycopg2-binary

In [1]:
def connect_to_db(host, database, user, password, port=5432):
    try:
        # Establish the connection
        conn = psycopg2.connect(
            host=host,
            database=database,
            user=user,
            password=password,
            port=port
        )
        print("Connection to the database was successful!")
        return conn
    except psycopg2.Error as e:
        print(f"Error: Could not make connection to the PostgreSQL database")
        print(e)
        return None


In [2]:
def get_password_from_file(file_path):
    try:
        with open(file_path, 'r') as file:
            password = file.read().strip()
        return password
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

In [15]:
import psycopg2
import json
from cryptography.fernet import Fernet

# Connect to PostgreSQL database
# Database connection parameters
host = 'data715-primary'
database = 'omop_covid'
user = 'aml14'
password = get_password_from_file('../db_pw.txt')

conn = connect_to_db(host, database, user, password)
cursor = conn.cursor()

Connection to the database was successful!


In [16]:
# Create tables for documents and key store
cursor.execute("""
CREATE TABLE IF NOT EXISTS key_store (
    doc_id TEXT PRIMARY KEY,
    key BYTEA
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS documents (
    doc_id TEXT PRIMARY KEY,
    content BYTEA
)
""")
conn.commit()

In [17]:
# Generate an encryption key
def generate_key():
    return Fernet.generate_key()

# Save encryption key in key store
def save_key(doc_id, key):
    cursor.execute("INSERT INTO key_store (doc_id, key) VALUES (%s, %s)", (doc_id, key))
    conn.commit()

# Retrieve encryption key from key store
def get_key(doc_id):
    cursor.execute("SELECT key FROM key_store WHERE doc_id = %s", (doc_id,))
    record = cursor.fetchone()
    return record[0] if record else None

# Add document to PostgreSQL with encryption
def add_document(doc_id, content):
    # Generate and store encryption key
    key = generate_key()
    save_key(doc_id, key)

    # Encrypt document content
    fernet = Fernet(key)
    encrypted_content = fernet.encrypt(json.dumps(content).encode())

    # Store encrypted document in PostgreSQL as bytea
    cursor.execute("INSERT INTO documents (doc_id, content) VALUES (%s, %s)", (doc_id, encrypted_content))
    conn.commit()

# Retrieve and decrypt document
def get_document(doc_id):
    # Get encryption key
    key = get_key(doc_id)
    if not key:
        return None

    # Retrieve encrypted document
    cursor.execute("SELECT content FROM documents WHERE doc_id = %s", (doc_id,))
    record = cursor.fetchone()
    if not record:
        return None

    # Ensure the token is in bytes format for decryption
    encrypted_content = bytes(record[0])  # Convert to bytes if it's not already

    # Decrypt document content
    fernet = Fernet(key)
    decrypted_content = fernet.decrypt(encrypted_content).decode()
    return json.loads(decrypted_content)

In [18]:
# Example usage
doc_id = "doc_1"
document_content = {"title": "Sensitive Document", "body": "This is a secure document."}

# Add document
add_document(doc_id, document_content)
print(f"Document '{doc_id}' added.")

# Retrieve document
retrieved_content = get_document(doc_id)
print("Retrieved Document Content:", retrieved_content)

Document 'doc_1' added.
Retrieved Document Content: {'title': 'Sensitive Document', 'body': 'This is a secure document.'}


In [23]:
import json
from cryptography.fernet import Fernet
import psycopg2

# Connect to PostgreSQL database
conn = connect_to_db(host, database, user, password)
cursor = conn.cursor()

# Function to create new tables for decrypted data and keys
def create_decrypted_tables():
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS documents_decrypt (
        doc_id TEXT PRIMARY KEY,
        content JSONB
    )
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS key_store_decrypt (
        doc_id TEXT PRIMARY KEY,
        key TEXT
    )
    """)
    conn.commit()

# Function to retrieve encryption key
def get_encryption_key(doc_id):
    cursor.execute("SELECT key FROM key_store WHERE doc_id = %s", (doc_id,))
    record = cursor.fetchone()
    # Convert memoryview to bytes if necessary
    return bytes(record[0]) if record else None

# Function to retrieve and decrypt document content
def decrypt_and_store_document(doc_id):
    # Retrieve encryption key from the original key store
    key = get_encryption_key(doc_id)
    if not key:
        print(f"No key found for document ID: {doc_id}")
        return

    # Retrieve encrypted document content
    cursor.execute("SELECT content FROM documents WHERE doc_id = %s", (doc_id,))
    record = cursor.fetchone()
    if not record:
        print(f"No document found for document ID: {doc_id}")
        return

    # Decrypt document content
    fernet = Fernet(key)
    encrypted_content = bytes(record[0])  # Convert memoryview to bytes
    decrypted_content = fernet.decrypt(encrypted_content).decode()

    # Convert decrypted content from JSON string to JSON object
    content_json = json.loads(decrypted_content)

    # Store decrypted data in documents_decrypt table
    cursor.execute("INSERT INTO documents_decrypt (doc_id, content) VALUES (%s, %s::jsonb)", (doc_id, json.dumps(content_json)))
    conn.commit()

    # Store decrypted key in key_store_decrypt table
    cursor.execute("INSERT INTO key_store_decrypt (doc_id, key) VALUES (%s, %s)", (doc_id, key.decode()))
    conn.commit()
    print(f"Decrypted data and key stored for document ID: {doc_id}")

# Create tables for decrypted data and keys
create_decrypted_tables()

# Decrypt and store document data
doc_id = "doc_1"  # Example document ID; replace with your actual document ID
decrypt_and_store_document(doc_id)

# Close the database connection
conn.close()


Connection to the database was successful!
Decrypted data and key stored for document ID: doc_1
